<a href="https://colab.research.google.com/github/peremartra/CH13/blob/main/CH12_NB02_arithmetic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CH12 · NB02 — Padding and KV cache arithmetic

Companion to section 12.7. Pure Python, no GPU required.

In [1]:
def padding_report(target_width, multiple=128):
    served = ((target_width + multiple - 1) // multiple) * multiple   #A
    return served, served - target_width

def snap_to_aligned(target_width, multiple=128):
    return (target_width // multiple) * multiple                      #B

raw = int(8192 * 0.80)                 # 6553, the naive 20% cut
served, waste = padding_report(raw)    # kernel actually runs 6656 wide
aligned = snap_to_aligned(raw)         # prune to 6528 instead, zero waste
print(f"naive target {raw}: served {served}, wasted columns {waste}")
print(f"aligned target {aligned}: served {aligned}, wasted columns 0")

naive target 6553: served 6656, wasted columns 103
aligned target 6528: served 6528, wasted columns 0


In [2]:
def kv_cache_bytes(L_attn, H_kv, d_head, seq_len, batch, dtype_bytes=2):
    return 2 * L_attn * H_kv * d_head * seq_len * batch * dtype_bytes

base = kv_cache_bytes(L_attn=28, H_kv=8, d_head=128, seq_len=2048, batch=64)
slim = kv_cache_bytes(L_attn=16, H_kv=8, d_head=128, seq_len=2048, batch=64)
print(f"cache after pruning is {slim / base:.1%} of the original")

cache after pruning is 57.1% of the original


In [3]:
small = kv_cache_bytes(L_attn=16, H_kv=8, d_head=128, seq_len=300, batch=8)
print(f"{small / 1e6:.2f} MB ({small / 2**20:.0f} MiB)")

157.29 MB (150 MiB)


## Section 12.9 — Proving it pays off

Cost model and latency report. The traces below are constructed stand-ins
(illustrative), not measurements.

In [4]:
def breakeven_requests_per_hour(gpu_cost_per_hour, api_cost_per_request):
    return gpu_cost_per_hour / api_cost_per_request

def slm_cost_per_request(gpu_cost_per_hour, requests_per_hour,
                         api_cost_per_request, escalation_rate):
    gpu_share = gpu_cost_per_hour / requests_per_hour
    return gpu_share + escalation_rate * api_cost_per_request

api_cost = 0.004
be = breakeven_requests_per_hour(gpu_cost_per_hour=2.50,
                                 api_cost_per_request=api_cost)
slm = slm_cost_per_request(
    gpu_cost_per_hour=2.50, requests_per_hour=2950,
    api_cost_per_request=api_cost, escalation_rate=0.042)
saving = 1 - slm / api_cost
print(f"break-even at {be:.0f} requests/hour")
print(f"SLM cost per request ${slm:.6f} ({saving:.1%} below the API)")

break-even at 625 requests/hour
SLM cost per request $0.001015 (74.6% below the API)


In [5]:
def latency_report(samples_ms):
    s = sorted(samples_ms)
    pct = lambda p: s[int(p / 100 * (len(s) - 1))]
    return pct(50), pct(95), pct(99)

slm_ms = [38] * 50 + [62] * 45 + [84] * 5
api_ms = [240] * 50 + [620] * 45 + [1150] * 5

print(f"{'':<13}{'p50 (ms)':>9}{'p95 (ms)':>9}{'p99 (ms)':>9}")
for name, samples in [("SLM", slm_ms), ("Frontier API", api_ms)]:
    p50, p95, p99 = latency_report(samples)
    print(f"{name:<13}{p50:>9,}{p95:>9,}{p99:>9,}")

              p50 (ms) p95 (ms) p99 (ms)
SLM                 38       62       84
Frontier API       240      620    1,150
